In [1]:
from astropy.table import Table
import pandas as pd
import numpy as np
from pandarallel import pandarallel
from concurrent.futures import ThreadPoolExecutor

In [2]:
# Initialize pandarallel to use all available CPUs (8 threads)
pandarallel.initialize(progress_bar=True, nb_workers=8)

INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.

https://nalepae.github.io/pandarallel/troubleshooting/


In [3]:

def load_votable_to_dataframe(file_path):

    # Read the VOTable file using astropy
    votable = Table.read(file_path, format='votable')

    # Convert to Pandas DataFrame
    df = votable.to_pandas()

    return df

# Example usage
file_path = "C:\\git_repo\\cool-dwarf_stellar_parameter_inference_from_survey_data\\data\\LAMOST_data\\LAMOST_dM0-9_Gaia_joined.vot"
df = load_votable_to_dataframe(file_path)


In [4]:
df.head()

,lamostmdwarf_oid,obsid,subclass,gaia_source_id,gaia_dr3_id,Plx,e_Plx,Gmag,BPmag,RPmag
0,97,565707070,dM2,1164942847486606592,1164942847486606592,4.957075,0.065747,16.924017,18.232756,15.793173
1,291,573407249,dM4,1377339941030819712,1377339941030819712,NaN,NaN,14.571911,15.722878,13.088904
2,807,594008071,dM4,390859792220346752,390859792220346752,8.889988,0.053381,16.375603,18.026264,15.145778
3,1197,604714183,dM2,2537044122414850432,2537044122414850432,1.546001,0.097267,17.442085,18.566982,16.411224
4,1341,606106195,dM1,626334771538326016,626334771538326016,2.160380,0.280078,17.190065,18.203493,16.023268


### Sanity check

In [5]:
print(df.shape[0])

796800


original small LAMOST df (M dwarf)

In [6]:
def load_votable_to_dataframe(file_path):
    # Read the VOTable file using astropy
    votable = Table.read(file_path, format='votable')

    df = votable.to_pandas()

    return df

file_path = "C:\\git_repo\\cool-dwarf_stellar_parameter_inference_from_survey_data\\data\\LAMOST_data\\LAMOST_dM0-9.vot"
df_LAMOST = load_votable_to_dataframe(file_path)
print(df_LAMOST.head())

   COLID_92974 COLID_92992          COLID_93001
0    559514185         dM0  1488665557766730752
1    559710175         dM4  4454322234930963456
2    562401159         dM1   804027502834099840
3    562401212         dM2   803949914248711552
4    562403056         dM0   805264625217416832


In [7]:
# Check the number of rows in the dataframe
num_rows = df_LAMOST.shape[0]
print("Number of rows:", num_rows)

Number of rows: 805806


In [8]:
# Check if a specific value is in the 'gaia_dr3_id' column
value_to_check = 4454322234930963456
is_value_present = value_to_check in df['gaia_dr3_id'].values
print(f"Is the value {value_to_check} in 'gaia_dr3_id' column?", is_value_present)

Is the value 4454322234930963456 in 'gaia_dr3_id' column? True


### Clean the Gaia X LAMOST_small data

In [9]:
# Drop rows where Parallax is missing or negative (bad data)
df_clean = df.dropna(subset=['obsid', 'Plx', 'Gmag', 'BPmag', 'RPmag'])
df_clean = df_clean[df_clean['Plx'] > 0]  # Parallax must be positive for distance

# Calculate Distance (in parsecs)
# Since Gaia Plx is in milliarcseconds, so we divide 1000 by Plx rather than 1/Plx
df_clean['distance_gaia_pc'] = 1000 / df_clean['Plx']

# Calculate Absolute G Magnitude (M_G)
# Formula: M = m - 5*log10(d) + 5
#df_clean['abs_Gmag'] = df_clean['Gmag'] - 5 * np.log10(df_clean['distance_gaia_pc']) + 5
#df_clean['abs_Bmag'] = df_clean['BPmag'] - 5 * np.log10(df_clean['distance_gaia_pc']) + 5
#df_clean['abs_Rmag'] = df_clean['RPmag'] - 5 * np.log10(df_clean['distance_pc']) + 5

#print(df_clean[['subclass', 'distance_pc', 'abs_Gmag', 'bp_rp']].head())
print(df_clean.head())

   lamostmdwarf_oid      obsid subclass       gaia_source_id  \
0                97  565707070      dM2  1164942847486606592   
2               807  594008071      dM4   390859792220346752   
3              1197  604714183      dM2  2537044122414850432   
4              1341  606106195      dM1   626334771538326016   
5              1585  616702039      dM0   423482985111866240   

           gaia_dr3_id       Plx     e_Plx       Gmag      BPmag      RPmag  \
0  1164942847486606592  4.957075  0.065747  16.924017  18.232756  15.793173   
2   390859792220346752  8.889988  0.053381  16.375603  18.026264  15.145778   
3  2537044122414850432  1.546001  0.097267  17.442085  18.566982  16.411224   
4   626334771538326016  2.160380  0.280078  17.190065  18.203493  16.023268   
5   423482985111866240  1.426469  0.075417  17.093874  18.109341  16.099749   

   distance_pc  
0   201.731885  
2   112.486085  
3   646.829934  
4   462.881467  
5   701.031687  


In [10]:
print(df_clean.shape[0])

773223


In [11]:
df_clean = df_clean.drop(columns=['lamostmdwarf_oid', 'e_Plx', 'Plx', 'gaia_dr3_id'])

In [12]:
df_clean.head()

,obsid,subclass,gaia_source_id,Gmag,BPmag,RPmag,distance_pc
0,565707070,dM2,1164942847486606592,16.924017,18.232756,15.793173,201.731885
2,594008071,dM4,390859792220346752,16.375603,18.026264,15.145778,112.486085
3,604714183,dM2,2537044122414850432,17.442085,18.566982,16.411224,646.829934
4,606106195,dM1,626334771538326016,17.190065,18.203493,16.023268,462.881467
5,616702039,dM0,423482985111866240,17.093874,18.109341,16.099749,701.031687


### Cross match df_clean with LAMOST and LAMOST_vac dataset

In [14]:
import pandas as pd
from astropy.table import Table
from concurrent.futures import ThreadPoolExecutor

# Input Paths
lamost_csv_path = r"C:\git_repo\cool-dwarf_stellar_parameter_inference_from_survey_data\data\LAMOST_data\LAMOST_dM0-9.csv"
lamost_fits_path = r"C:\git_repo\cool-dwarf_stellar_parameter_inference_from_survey_data\data\LAMOST_fits_data\dr8_final_vac_flag_parameter.fits"

# Helper functions for reading
def load_csv(path):
    return pd.read_csv(path)

def load_fits(path):
    return Table.read(path, format='fits').to_pandas()

# Load datasets in parallel (Exploiting threading for I/O bound tasks)
print("Loading datasets in parallel with ThreadPoolExecutor...")
with ThreadPoolExecutor(max_workers=8) as executor:
    future_csv = executor.submit(load_csv, lamost_csv_path)
    future_fits = executor.submit(load_fits, lamost_fits_path)
    
    df_LAMOST = future_csv.result()
    df_LAMOST_vac = future_fits.result()

print("Datasets loaded successfully.")
print(f"df_LAMOST shape: {df_LAMOST.shape}")
print(f"df_LAMOST_vac shape: {df_LAMOST_vac.shape}")

Loading datasets in parallel with ThreadPoolExecutor...
Datasets loaded successfully.
df_LAMOST shape: (696035, 17)
df_LAMOST_vac shape: (7109030, 112)


In [15]:
print(df_LAMOST.columns.tolist())

['obsid', 'spid', 'snrr', 'snri', 'class', 'subclass', 'ps_id', 'mag_ps_g', 'mag_ps_r', 'mag_ps_i', 'mag_ps_z', 'mag_ps_y', 'gaia_source_id', 'gaia_g_mean_mag', 'teff', 'logg', 'feh']


In [16]:
print(df_LAMOST_vac.columns.tolist())

['obsid', 'ra_obs', 'dec_obs', 'FEH_APOGEE', 'MH_APOGEE', 'CH_APOGEE', 'NH_APOGEE', 'CFE_APOGEE', 'NFE_APOGEE', 'AFE_APOGEE', 'LOGG_APOGEE', 'TEFF_PASTEL', 'LOGG_PASTEL', 'FEH_PASTEL', 'A_GG', 'A_BP', 'A_RP', 'A_J', 'A_H', 'A_KS', 'A_W1', 'A_W2', 'A_BAP', 'A_VAP', 'A_RAP', 'A_GSD', 'A_RSD', 'A_ISD', 'snrg', 'rv', 'err_rv', 'err_teff_pastel', 'err_feh_pastel', 'err_logg_pastel', 'err_feh_apogee', 'err_mh_apogee', 'err_afe_apogee', 'err_cfe_apogee', 'err_nfe_apogee', 'err_logg_apogee', 'err_a_gg', 'err_a_bp', 'err_a_rp', 'a_bprp', 'err_a_bprp', 'err_a_j', 'err_a_h', 'err_a_ks', 'err_a_bap', 'err_a_vap', 'err_a_rap', 'err_a_gsd', 'err_a_rsd', 'err_a_isd', 'err_a_w1', 'err_a_w2', 'mh_afe', 'err_mh_afe', 'GAIAID', 'EBV', 'uqflag', 'dist_phot', 'err_dist_phot', 'x', 'err_x', 'y', 'err_y', 'z', 'err_z', 'r', 'err_r', 'phi', 'err_phi', 'gl', 'gb', 'u', 'v', 'w', 'e_u', 'e_v', 'e_w', 'vr', 'vphi', 'vz', 'e_vr', 'e_vphi', 'e_vz', 'feh_vmp', 'err_feh_vmp', 'flag_feh_apogee', 'flag_mh_apogee', 'fl

In [34]:
# Define columns to keep for df_LAMOST_vac
columns_to_keep_vac = ['obsid', 
                       # all absolute magnitudes
                       'A_GG', 'A_BP', 'A_RP', # Gaia EDR3
                       'A_J', 'A_H', 'A_KS', # 2MASS
                       'A_W1', 'A_W2', # WISE
                       'A_BAP', 'A_VAP', 'A_RAP', #APASS
                       'A_GSD', 'A_RSD', 'A_ISD']  # SDSS

# Define columns to drop for df_LAMOST
irrelevant_columns_lamost = ['spid', 'snrr','snri', 'class','subclass', 'ps_id','gaia_source_id', 'gaia_g_mean_mag',
                             'feh' # all NA in this column
                             ] 

# Define columns to check for missing values
# Rows with NaNs in these columns will be dropped
nan_columns_check_lamost = [] 
nan_columns_check_vac = []

# Apply Cleaning
if irrelevant_columns_lamost:
    df_LAMOST.drop(columns=irrelevant_columns_lamost, inplace=True, errors='ignore')

if columns_to_keep_vac:
    df_LAMOST_vac = df_LAMOST_vac[columns_to_keep_vac]

# Drop rows with specific value (-999) in specified columns
columns_to_check_for_value = ['mag_ps_y', 'mag_ps_z', 'mag_ps_i', 'mag_ps_r', 'mag_ps_g'] # apparent magnitudes
for col in columns_to_check_for_value:
    if col in df_LAMOST.columns:
        df_LAMOST = df_LAMOST[df_LAMOST[col] != -999]

if nan_columns_check_lamost:
    df_LAMOST.dropna(subset=nan_columns_check_lamost, inplace=True)

if nan_columns_check_vac:
    df_LAMOST_vac.dropna(subset=nan_columns_check_vac, inplace=True)

print("Cleaning complete")

Cleaning complete (based on placeholders).


In [38]:
df_LAMOST_vac.head(10)

,obsid,A_GG,A_BP,A_RP,A_J,A_H,A_KS,A_W1,A_W2,A_BAP,A_VAP,A_RAP,A_GSD,A_RSD,A_ISD
0,105704216.0,3.751406,3.881304,2.902264,2.336192,2.156372,1.663003,2.524816,1.010052,3.043267,3.874843,2.984920,2.919531,3.610125,3.552638
1,814404042.0,4.068060,4.360425,4.942953,3.212042,4.168449,2.917255,3.933477,3.162223,6.281215,4.351768,4.687097,4.706560,4.050163,6.133793
2,309907119.0,3.817540,3.999802,2.873478,2.580539,1.899501,2.074955,2.280157,1.881451,4.485433,3.520332,3.793094,4.381365,3.434602,3.553047
3,564506028.0,4.082883,4.700488,3.546080,2.831862,3.216836,2.827840,2.193428,2.779294,5.195091,4.152187,4.562494,4.246381,4.444151,4.589468
4,557708087.0,1.277164,1.984290,0.647106,0.245124,-0.012795,-0.667526,0.023370,-0.435779,2.089422,1.887725,1.910387,2.135766,1.395873,1.528985
5,228605225.0,4.908501,5.517083,4.153411,4.032901,3.515193,3.371768,3.212000,3.304538,6.058166,4.773209,4.693232,5.233153,4.242326,4.134130
6,507005148.0,4.721043,5.804685,4.395596,4.018856,3.903585,3.934841,3.216208,3.275542,6.903694,5.404795,4.855844,4.739204,4.408143,4.957470
7,756711092.0,3.068360,4.554669,2.173767,1.954524,1.686570,1.184146,1.316306,2.600422,3.668927,3.549033,3.549498,2.940325,3.033015,1.547707
8,564508230.0,2.538508,4.394424,3.679119,2.047662,2.201259,2.897748,1.874365,1.385151,5.182198,3.290172,3.144165,3.549851,2.390765,3.044298
9,414402010.0,4.645375,5.539663,4.799868,3.656878,3.637173,3.818758,2.607846,3.200141,5.749143,4.662692,5.437180,4.404426,4.987878,4.183251


In [39]:
df_LAMOST.head(10)

,obsid,mag_ps_g,mag_ps_r,mag_ps_i,mag_ps_z,mag_ps_y,teff,logg
3,256305177,19.075001,17.819201,16.986500,16.596300,16.408300,3711.42,4.479
6,258008147,17.192600,15.963200,15.217900,14.883200,14.696400,3788.46,4.624
12,265106071,16.774599,15.635600,14.321200,13.731500,13.446200,3571.49,4.762
17,266710213,18.988501,17.788700,16.951900,16.555300,16.377701,3613.57,4.548
18,266711235,18.396799,17.197201,16.545799,16.242901,16.066099,3933.65,4.626
19,266714026,18.702900,17.493500,16.857901,16.577400,16.375700,3959.34,4.569
20,266716042,18.511200,17.368900,16.689100,16.378599,16.208599,3866.20,4.695
30,267810110,17.290400,16.118500,15.125800,14.680300,14.467900,3522.99,4.682
31,268015094,16.336700,15.179800,13.952400,13.376000,13.105500,3341.43,4.778
34,269812080,18.910000,17.660801,16.470900,15.944500,15.672300,3510.48,4.694


In [43]:
print(df_LAMOST.shape[0])

229659


In [40]:
print(df_LAMOST_vac.shape[0])

7109030


### Sanity check on df_LAMOST and df_LAMOST_vac

--- Descriptive Statistics for Apparent Magnitudes (df_LAMOST) ---
            mag_ps_g       mag_ps_r       mag_ps_i       mag_ps_z  \
count  226933.000000  226933.000000  226933.000000  226933.000000   
mean       17.807854      16.578739      15.657473      15.224606   
std         1.072613       1.047763       1.031893       1.044694   
min        11.727000      10.614000       9.880000       9.345000   
25%        17.145800      15.934100      15.012700      14.570400   
50%        18.041500      16.812500      15.859000      15.404900   
75%        18.652901      17.414200      16.443399      16.010900   
max        26.930000      21.937901      22.003700      21.240601   

            mag_ps_y  
count  226933.000000  
mean       14.999983  
std         1.057830  
min         9.109000  
25%        14.342500  
50%        15.171400  
75%        15.789600  
max        20.577400  

--- Rows with negative apparent magnitudes (potential issues or very bright objects?) ---
mag_ps_g: No 

In [47]:
# Sanity Check for df_LAMOST_vac (Absolute Magnitudes)
print("--- Descriptive Statistics for Absolute Magnitudes (df_LAMOST_vac) ---")
print(df_LAMOST_vac[abs_mag_cols].describe())

# Check for missing values
print("\n--- Missing Values Count in Absolute Magnitude columns ---")
print(df_LAMOST_vac[abs_mag_cols].isnull().sum())

# Check mean values
print("\n--- Mean values ---")
print(df_LAMOST_vac[abs_mag_cols].mean())

--- Descriptive Statistics for Absolute Magnitudes (df_LAMOST_vac) ---
               A_GG          A_BP          A_RP           A_J           A_H  \
count  7.108884e+06  7.108884e+06  7.108884e+06  7.108884e+06  7.108884e+06   
mean   3.467871e+00  5.295861e+00  5.035653e+00  3.837934e+00  3.442544e+00   
std    5.627366e+02  6.304450e+02  7.607076e+02  4.924130e+02  5.196627e+02   
min   -1.421740e+06 -9.118463e+05 -6.907703e+03 -2.768361e+03 -7.305480e+04   
25%    2.601807e+00  3.106667e+00  2.357153e+00  1.849288e+00  1.639340e+00   
50%    3.846676e+00  4.322493e+00  3.513322e+00  2.982483e+00  2.728987e+00   
75%    5.036415e+00  5.634186e+00  4.676504e+00  4.009418e+00  3.671449e+00   
max    1.849633e+05  8.803761e+05  1.235520e+06  6.366154e+05  9.236934e+05   

               A_KS          A_W1          A_W2         A_BAP         A_VAP  \
count  7.108884e+06  7.108884e+06  7.108884e+06  7.108884e+06  7.108884e+06   
mean   2.925194e+00  3.548173e+00  2.818137e+00  5.878105e+

In [50]:
# Drop rows where any column has a missing value in df_LAMOST and df_LAMOST_vac
print("Dropping rows with missing values in df_LAMOST and df_LAMOST_vac...")

# Drop rows with missing values in df_LAMOST
df_LAMOST_cleaned = df_LAMOST.dropna()
print(f"df_LAMOST: Dropped {df_LAMOST.shape[0] - df_LAMOST_cleaned.shape[0]} rows with missing values.")

# Drop rows with missing values in df_LAMOST_vac
df_LAMOST_vac_cleaned = df_LAMOST_vac.dropna()
print(f"df_LAMOST_vac: Dropped {df_LAMOST_vac.shape[0] - df_LAMOST_vac_cleaned.shape[0]} rows with missing values.")

# Display the shapes of the cleaned dataframes
print("Cleaned DataFrame Shapes:")
print(f"df_LAMOST_cleaned: {df_LAMOST_cleaned.shape}")
print(f"df_LAMOST_vac_cleaned: {df_LAMOST_vac_cleaned.shape}")

Dropping rows with missing values in df_LAMOST and df_LAMOST_vac...
df_LAMOST: Dropped 2726 rows with missing values.
df_LAMOST_vac: Dropped 146 rows with missing values.
Cleaned DataFrame Shapes:
df_LAMOST_cleaned: (226933, 8)
df_LAMOST_vac_cleaned: (7108884, 15)


In [ ]:
print("Merging df_clean with df_LAMOST")
# First Inner Join
df_merged_1 = df_clean.merge(df_LAMOST_cleaned, how='inner', on='obsid') 

In [ ]:
print("Merging with df_LAMOST_vac")
# Second inner Join
df_final = df_merged_1.merge(df_LAMOST_vac_cleaned, how='inner', on='obsid')

In [33]:
print(df_final.columns.tolist())

['obsid', 'subclass', 'gaia_source_id', 'Gmag', 'BPmag', 'RPmag', 'distance_gaia_pc', 'mag_ps_g', 'mag_ps_r', 'mag_ps_i', 'mag_ps_z', 'mag_ps_y', 'teff', 'logg', 'A_GG', 'A_BP', 'A_RP', 'A_J', 'A_H', 'A_KS', 'A_W1', 'A_W2', 'A_BAP', 'A_VAP', 'A_RAP', 'A_GSD', 'A_RSD', 'A_ISD']


In [32]:
output_path = r"C:\git_repo\cool-dwarf_stellar_parameter_inference_from_survey_data\data\Xmatch_gaia_LAMOST_LAMOSTvac.csv"
df_final.to_csv(output_path, index=False)
print(f"df_final saved to {output_path}")

df_final saved to C:\git_repo\cool-dwarf_stellar_parameter_inference_from_survey_data\data\Xmatch_gaia_LAMOST_LAMOSTvac.csv
